In [ ]:
# Cell 1 - run information (required)
import os, subprocess
REPO_DIR = os.environ.get("REPO_DIR", "")          # e.g. /kaggle/working/data266-lab2-team48
GITHUB_REPO = "Manjot0005/data266-lab2-team48"
BRANCH = "p2-finetune"
try:                                               # Colab: keep repo + outputs on Google Drive so a disconnect loses nothing
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/" + GITHUB_REPO.split("/")[-1]
    if not os.path.isdir(REPO_DIR):
        tok = userdata.get("GITHUB_TOKEN").strip()   # strip stray spaces/newlines from the pasted secret
        r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, f"https://{tok}@github.com/{GITHUB_REPO}.git", REPO_DIR],
                           capture_output=True, text=True)
        print("git clone:", "OK" if r.returncode == 0 else "FAILED (check GITHUB_TOKEN secret / repo name / branch)")
except Exception as e:                             # not on Colab, or GITHUB_TOKEN secret missing
    print("Colab/Drive setup skipped:", type(e).__name__)
if REPO_DIR and os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                                  # run from the repo root
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

# Part 2 — Fine-Tuning Stable Diffusion 1.5 with LoRA (Naruto BLIP Captions)

**Pipeline:** clean data → 90/10 split (seed 266) → two LoRA runs with the same data and step budget
(rank 4 vs rank 32) → 20 fixed prompts × 2 seeds × {base, run 1, run 2} → KID, CLIP score, LPIPS diversity,
blinded human evaluation, memorization check → push adapters, model card and images to Hugging Face.

Run on a single 16 GB GPU (Kaggle T4/P100 or Colab T4) with internet on.
Every number in the report must come from an output of this notebook.

In [ ]:
# Install pinned versions (torch/torchvision are preinstalled on Kaggle/Colab)
%pip install -q matplotlib pandas pillow tqdm diffusers==0.31.0 transformers==4.46.3 accelerate==1.1.1 peft==0.13.2 datasets==3.1.0 huggingface_hub==0.26.2 safetensors==0.4.5 "torchmetrics[image]==1.6.0" torch-fidelity==0.3.0 open_clip_torch==2.29.0 lpips==0.1.4

In [ ]:
import json, time, math, hashlib, io, gc
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageOps
from tqdm.auto import tqdm
import diffusers, transformers, peft, datasets, torchmetrics
print({"diffusers": diffusers.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
       "datasets": datasets.__version__, "torchmetrics": torchmetrics.__version__})

# ---------------- configuration ----------------
TEAM = "48"                                         # two-digit team number
HF_USER = "manjottxkaur"
DATASET_ID = "lambdalabs/naruto-blip-captions"
BASE_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"
RES = 512

RUNS = {   # same data, same step budget, same batch size and LR; only the LoRA rank differs
    "run1": dict(rank=4,  alpha=4,  lr=1e-4),
    "run2": dict(rank=32, alpha=32, lr=1e-4),
}
TRAIN_STEPS = 2000
BATCH_SIZE = 4
WARMUP_STEPS = 100
SAVE_EVERY = 500                                    # resumable checkpoints
GEN_STEPS, GUIDANCE = 30, 7.5                       # identical sampler settings for all models
GEN_SEEDS = [0, 1]

OUT = Path("outputs"); DATA_DIR = OUT / "data"; GEN_DIR = OUT / "generations"
PROMPTS_PATH = Path("prompts/p2_prompts.json")
EVAL_DIR = Path("eval"); HE_DIR = Path("human_eval")
for d in [OUT, DATA_DIR, GEN_DIR, EVAL_DIR, HE_DIR, PROMPTS_PATH.parent]:
    d.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "A GPU is required for Part 2"

def hf_repo(name): return f"{HF_USER}/data266-lab2-team{TEAM}-p2-{name}"
MODEL_REPOS = {"run1": hf_repo("run1-lora-r4"), "run2": hf_repo("run2-lora-r32")}
IMAGE_DATASET_REPO = hf_repo("images")

In [ ]:
# Hugging Face login: HF_TOKEN env var, Kaggle/Colab secret, or a cached `huggingface-cli login`
def get_hf_token():
    tok = os.environ.get("HF_TOKEN")
    if tok: return tok
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception: pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN").strip()
    except Exception: pass
    try:                                            # RunPod / local: token saved by `huggingface-cli login`
        from huggingface_hub import get_token
        return get_token()
    except Exception: pass
    return None

from huggingface_hub import login, HfApi, create_repo, DatasetCard
HF_TOKEN = get_hf_token()
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face as", HfApi().whoami()["name"])
else:
    print("WARNING: no HF_TOKEN found -- uploads will be skipped")

## 2.1 Data preparation
Load the raw bytes so that corrupted files can be detected, then remove corrupted images,
exact duplicates (MD5 of decoded pixels) and near-duplicates (identical 64-bit difference hash).

In [ ]:
from datasets import load_dataset, Image as HFImage, Dataset

raw = load_dataset(DATASET_ID, split="train").cast_column("image", HFImage(decode=False))
print("Raw images:", len(raw))
try:
    print("Dataset license (from dataset card):", DatasetCard.load(DATASET_ID).data.to_dict().get("license"))
except Exception as e:
    print("Could not read dataset card:", e)

def dhash(img, size=8):
    g = img.convert("L").resize((size + 1, size), Image.BILINEAR)
    a = np.asarray(g, dtype=np.int16)
    return np.packbits((a[:, 1:] > a[:, :-1]).flatten()).tobytes()

kept, n_corrupt, n_exact, n_near = [], 0, 0, 0
seen_md5, seen_dh = set(), set()
for ex in tqdm(raw, desc="cleaning"):
    b = ex["image"]["bytes"]
    try:
        Image.open(io.BytesIO(b)).verify()               # structural check
        img = Image.open(io.BytesIO(b)).convert("RGB")   # full decode
        if min(img.size) < 64: raise ValueError("too small")
    except Exception:
        n_corrupt += 1; continue
    md5 = hashlib.md5(img.tobytes()).hexdigest()
    if md5 in seen_md5: n_exact += 1; continue
    dh = dhash(img)
    if dh in seen_dh: n_near += 1; continue
    seen_md5.add(md5); seen_dh.add(dh)
    kept.append({"image_bytes": b, "text": ex["text"], "orig_w": img.size[0], "orig_h": img.size[1]})

cleaning_report = {"raw": len(raw), "corrupted_removed": n_corrupt, "exact_duplicates_removed": n_exact,
                   "near_duplicates_removed": n_near, "kept": len(kept)}
print(cleaning_report)

In [ ]:
# 90% train / 10% test with seed 266 -- test images are used ONLY for evaluation
clean = Dataset.from_list(kept)
split = clean.train_test_split(test_size=0.1, seed=SEED)
train_ds, test_ds = split["train"], split["test"]
print({"train": len(train_ds), "test": len(test_ds)})

def to_square(img, res=RES):
    return ImageOps.fit(img.convert("RGB"), (res, res), method=Image.BICUBIC, centering=(0.5, 0.5))

def export(ds, name):
    d = DATA_DIR / name; d.mkdir(exist_ok=True)
    rows = []
    for i, ex in enumerate(tqdm(ds, desc=f"resize {name}")):
        fn = f"{name}_{i:05d}.png"
        to_square(Image.open(io.BytesIO(ex["image_bytes"]))).save(d / fn)
        rows.append({"file": str(d / fn), "text": ex["text"]})
    pd.DataFrame(rows).to_csv(DATA_DIR / f"{name}.csv", index=False)
    return pd.DataFrame(rows)

train_df, test_df = export(train_ds, "train"), export(test_ds, "test")

In [ ]:
# Dataset description (numbers for the report)
cap_len = train_df.text.str.split().str.len()
sizes = pd.Series([f"{w}x{h}" for w, h in zip(clean["orig_w"], clean["orig_h"])]).value_counts()
desc = {"n_train": len(train_df), "n_test": len(test_df),
        "caption_words_mean": round(float(cap_len.mean()), 2), "caption_words_min": int(cap_len.min()),
        "caption_words_max": int(cap_len.max()), "most_common_original_sizes": sizes.head(3).to_dict()}
print(desc)
words = pd.Series(" ".join(train_df.text.str.lower()).split()).value_counts()
print("Most frequent caption words:", words.head(25).to_dict())

fig, axes = plt.subplots(2, 6, figsize=(18, 6.5))
for ax, (_, r) in zip(axes.flat, train_df.sample(12, random_state=SEED).iterrows()):
    ax.imshow(Image.open(r.file)); ax.set_title(r.text[:40], fontsize=8); ax.axis("off")
plt.suptitle("Training samples (512x512)"); plt.tight_layout(); plt.show()

**Dataset description (write in your own words for the report).** Use the numbers printed above.
Two limitations/biases to discuss — look at the samples and word counts and decide, e.g.: captions are
automatic BLIP captions (short, generic, sometimes wrong — they rarely name characters or the art style);
the character distribution is uneven (a few main characters dominate); images are single-source anime
screenshots, so the model will impose that style and has little variety in pose/background.

## 2.3 (part a) The 20 fixed prompts — write and COMMIT them before training ends
10 in-domain prompts are test-set captions (sampled with seed 266); 5 out-of-domain and 5 compositional prompts
are written by us. **Edit the out-of-domain / compositional lists if you want, then commit
`prompts/p2_prompts.json` and push right away.** If the file already exists it is loaded unchanged,
so the committed version is the one used.

In [ ]:
OOD_PROMPTS = [
    "a photograph of a red sports car on a mountain road",
    "a bowl of ramen on a wooden table, studio food photography",
    "a watercolor painting of a lighthouse at sunset",
    "an astronaut riding a horse on the moon",
    "a modern city skyline at night with neon lights",
]
COMP_PROMPTS = [
    "two characters, one in red and one in blue",
    "a girl with pink hair standing next to a boy with black hair",
    "a man holding a red umbrella and a woman holding a yellow umbrella",
    "a small cat on the left and a large dog on the right",
    "three people sitting on a bench, the one in the middle wears glasses",
]
if PROMPTS_PATH.exists():
    PROMPTS = json.loads(PROMPTS_PATH.read_text())
    print("Loaded committed prompts from", PROMPTS_PATH)
else:
    in_domain = random.Random(SEED).sample(sorted(set(test_df.text)), 10)
    PROMPTS = ([{"id": f"ID{i+1:02d}", "category": "in_domain", "prompt": p} for i, p in enumerate(in_domain)] +
               [{"id": f"OOD{i+1:02d}", "category": "out_of_domain", "prompt": p} for i, p in enumerate(OOD_PROMPTS)] +
               [{"id": f"CMP{i+1:02d}", "category": "compositional", "prompt": p} for i, p in enumerate(COMP_PROMPTS)])
    PROMPTS_PATH.write_text(json.dumps(PROMPTS, indent=2))
    print("Wrote", PROMPTS_PATH, "-> commit and push this file now (git add prompts/p2_prompts.json)")
assert len(PROMPTS) == 20
pd.DataFrame(PROMPTS)

## 2.2 Fine-tuning — two LoRA runs on the UNet
Both runs: same cleaned training split, same 2,000 steps × batch 4, same LR and schedule; run 1 uses LoRA rank 4,
run 2 uses LoRA rank 32 on the attention projections (`to_q, to_k, to_v, to_out.0`).
VAE latents and CLIP text embeddings are pre-computed once (the VAE and text encoder are frozen), which makes
training about 2× faster on a T4. Mixed precision (fp16 autocast, fp32 LoRA weights) + gradient checkpointing.
Training is resumable from the last checkpoint if the session restarts.

In [ ]:
from diffusers import (StableDiffusionPipeline, DDPMScheduler, AutoencoderKL, UNet2DConditionModel,
                       DPMSolverMultistepScheduler)
from diffusers.optimization import get_scheduler
from diffusers.training_utils import cast_training_params
from diffusers.utils import convert_state_dict_to_diffusers
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict, set_peft_model_state_dict

CACHE = OUT / "train_cache.pt"
if CACHE.exists():
    cache = torch.load(CACHE)
else:
    tok = CLIPTokenizer.from_pretrained(BASE_MODEL, subfolder="tokenizer")
    te = CLIPTextModel.from_pretrained(BASE_MODEL, subfolder="text_encoder").to(device, torch.float16).eval()
    vae = AutoencoderKL.from_pretrained(BASE_MODEL, subfolder="vae").to(device).eval()   # fp32 for stable encoding
    lat, emb = [], []
    torch.backends.cudnn.enabled = False   # workaround: cuDNN illegal-memory-access in VAE encode on some GPUs
    with torch.no_grad():
        for i in tqdm(range(0, len(train_df), 2), desc="pre-compute latents"):
            chunk = train_df.iloc[i:i + 2]
            px = torch.stack([torch.from_numpy(np.asarray(Image.open(f), dtype=np.float32) / 127.5 - 1).permute(2, 0, 1)
                              for f in chunk.file]).to(device)
            lat.append((vae.encode(px).latent_dist.sample() * vae.config.scaling_factor).half().cpu())
            ids = tok(list(chunk.text), padding="max_length", max_length=tok.model_max_length,
                      truncation=True, return_tensors="pt").input_ids.to(device)
            emb.append(te(ids)[0].half().cpu())
    torch.backends.cudnn.enabled = True
    cache = {"latents": torch.cat(lat), "emb": torch.cat(emb)}
    torch.save(cache, CACHE)
    del te, vae; gc.collect(); torch.cuda.empty_cache()
print({k: tuple(v.shape) for k, v in cache.items()})

In [ ]:
def train_lora(run_name, rank, alpha, lr, steps=TRAIN_STEPS, bs=BATCH_SIZE, seed=SEED):
    out_dir = OUT / run_name; out_dir.mkdir(exist_ok=True)
    final = out_dir / "final"
    if (final / "metrics.json").exists():
        print(run_name, "already trained"); return json.loads((final / "metrics.json").read_text())
    torch.manual_seed(seed)
    noise_sched = DDPMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler")
    unet = UNet2DConditionModel.from_pretrained(BASE_MODEL, subfolder="unet", torch_dtype=torch.float16).to(device)
    unet.requires_grad_(False)
    unet.add_adapter(LoraConfig(r=rank, lora_alpha=alpha, init_lora_weights="gaussian",
                                target_modules=["to_k", "to_q", "to_v", "to_out.0"]))
    cast_training_params(unet, dtype=torch.float32)          # LoRA weights in fp32, frozen base in fp16
    unet.enable_gradient_checkpointing(); unet.train()
    params = [p for p in unet.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in params)
    n_total = sum(p.numel() for p in unet.parameters())
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-2)
    sched = get_scheduler("cosine", optimizer=opt, num_warmup_steps=WARMUP_STEPS, num_training_steps=steps)
    scaler = torch.amp.GradScaler("cuda")

    lat_all, emb_all = cache["latents"], cache["emb"]
    N = lat_all.shape[0]
    step, elapsed, losses = 0, 0.0, []
    ckpt = out_dir / "ckpt.pt"
    if ckpt.exists():                                        # resume after a session restart
        s = torch.load(ckpt, weights_only=False)
        set_peft_model_state_dict(unet, s["lora"]); cast_training_params(unet, dtype=torch.float32)
        opt.load_state_dict(s["opt"]); sched.load_state_dict(s["sched"]); scaler.load_state_dict(s["scaler"])
        step, elapsed, losses = s["step"], s["elapsed"], s["losses"]
        print(f"Resumed {run_name} at step {step}")

    g = torch.Generator().manual_seed(seed)
    order = torch.cat([torch.randperm(N, generator=g) for _ in range(math.ceil(steps * bs / N) + 1)])
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    t0 = time.time()
    pbar = tqdm(total=steps, initial=step, desc=run_name)
    while step < steps:
        idx = order[step * bs:(step + 1) * bs]
        lat = lat_all[idx].to(device, torch.float32)
        flip = (torch.rand(len(idx), generator=g) < 0.5).to(device)         # horizontal flip augmentation in latent space
        lat[flip] = torch.flip(lat[flip], dims=[3])
        enc = emb_all[idx].to(device)
        noise = torch.randn_like(lat)
        t = torch.randint(0, noise_sched.config.num_train_timesteps, (len(idx),), device=device).long()
        noisy = noise_sched.add_noise(lat, noise, t)
        with torch.autocast("cuda", dtype=torch.float16):
            pred = unet(noisy.half(), t, enc).sample
        loss = torch.nn.functional.mse_loss(pred.float(), noise)   # SD 1.5 predicts epsilon
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        losses.append(loss.item()); step += 1; pbar.update(1)
        if step % 50 == 0: pbar.set_postfix(loss=f"{np.mean(losses[-50:]):.4f}")
        if step % SAVE_EVERY == 0 or step == steps:
            torch.cuda.synchronize()
            torch.save({"lora": get_peft_model_state_dict(unet), "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "scaler": scaler.state_dict(), "step": step, "elapsed": elapsed + time.time() - t0,
                        "losses": losses}, ckpt)
    pbar.close(); torch.cuda.synchronize()
    elapsed += time.time() - t0
    final.mkdir(exist_ok=True)
    StableDiffusionPipeline.save_lora_weights(
        save_directory=final, unet_lora_layers=convert_state_dict_to_diffusers(get_peft_model_state_dict(unet)),
        safe_serialization=True)
    metrics = {"run": run_name, "method": f"LoRA r={rank}, alpha={alpha}", "trainable_params": n_trainable,
               "trainable_pct": round(100 * n_trainable / n_total, 3), "learning_rate": lr, "steps": steps,
               "batch_size": bs, "peak_gpu_mem_GB": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
               "train_time_min": round(elapsed / 60, 1), "gpu_hours": round(elapsed / 3600, 3),
               "gpu": torch.cuda.get_device_name(0), "final_loss_avg_last100": round(float(np.mean(losses[-100:])), 4)}
    (final / "metrics.json").write_text(json.dumps(metrics, indent=2))
    (final / "losses.json").write_text(json.dumps(losses))
    del unet, opt; gc.collect(); torch.cuda.empty_cache()
    print(metrics)
    return metrics

In [ ]:
train_metrics = {name: train_lora(name, **cfg) for name, cfg in RUNS.items()}

In [ ]:
# Training table for the report (all values measured above)
train_table = pd.DataFrame(train_metrics.values()).set_index("run")
train_table.to_csv(EVAL_DIR / "p2_training.csv")
display(train_table[["method", "trainable_params", "trainable_pct", "learning_rate", "steps", "batch_size",
                     "peak_gpu_mem_GB", "train_time_min", "gpu_hours"]])

plt.figure(figsize=(8, 3.5))
for name in RUNS:
    l = json.loads((OUT / name / "final" / "losses.json").read_text())
    plt.plot(pd.Series(l).rolling(50).mean(), label=f"{name} (rank {RUNS[name]['rank']})")
plt.xlabel("step"); plt.ylabel("MSE loss (50-step mean)"); plt.legend(); plt.title("Training loss"); plt.show()

In [ ]:
# Push both adapters to Hugging Face (model card for the final run is written in section 2.4)
api = HfApi()
if HF_TOKEN:
    for name, repo in MODEL_REPOS.items():
        create_repo(repo, repo_type="model", exist_ok=True)
        api.upload_folder(folder_path=str(OUT / name / "final"), repo_id=repo, repo_type="model",
                          commit_message=f"{name} LoRA adapter")
        print("pushed", f"https://huggingface.co/{repo}")

## 2.3 Image generation — 20 prompts × seeds {0, 1} × {base, run 1, run 2} = 120 images
Same sampler for every model: DPM-Solver++ multistep, 30 steps, guidance 7.5, 512×512.
The built-in Stable Diffusion safety checker is kept **on** as the NSFW output filter; flags are recorded.

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL, torch_dtype=torch.float16).to(device)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)
assert pipe.safety_checker is not None, "keep the NSFW safety checker enabled"

def generate_all(model_name):
    rows, d = [], GEN_DIR / model_name
    d.mkdir(exist_ok=True)
    if model_name != "base":
        pipe.load_lora_weights(str(OUT / model_name / "final"), weight_name="pytorch_lora_weights.safetensors")
    for p in tqdm(PROMPTS, desc=model_name):
        for seed in GEN_SEEDS:
            fn = d / f"{p['id']}_s{seed}.png"
            torch.cuda.synchronize(); t0 = time.time()
            out = pipe(p["prompt"], num_inference_steps=GEN_STEPS, guidance_scale=GUIDANCE, height=RES, width=RES,
                       generator=torch.Generator(device).manual_seed(seed))
            torch.cuda.synchronize()
            out.images[0].save(fn)
            rows.append({"file_name": f"{model_name}/{fn.name}", "model": model_name, "prompt_id": p["id"],
                         "category": p["category"], "prompt": p["prompt"], "seed": seed, "steps": GEN_STEPS,
                         "guidance": GUIDANCE, "sampler": "DPMSolverMultistep",
                         "nsfw_flagged": bool(out.nsfw_content_detected[0]), "seconds": round(time.time() - t0, 2)})
    if model_name != "base":
        pipe.unload_lora_weights()
    return rows

gen_rows = []
for m in ["base", "run1", "run2"]:
    gen_rows += generate_all(m)
gen_df = pd.DataFrame(gen_rows)
gen_df.to_csv(GEN_DIR / "metadata.csv", index=False)
print(gen_df.groupby("model").agg(images=("file_name", "count"), nsfw_flagged=("nsfw_flagged", "sum"),
                                  median_sec_per_image=("seconds", "median")))

In [ ]:
# Side-by-side grids for 8 prompts (base vs run 1 vs run 2, seed 0)
def load_gen(model, pid, seed): return Image.open(GEN_DIR / model / f"{pid}_s{seed}.png")
grid_ids = [p["id"] for p in PROMPTS if p["category"] == "in_domain"][:4] + \
           [p["id"] for p in PROMPTS if p["category"] == "out_of_domain"][:2] + \
           [p["id"] for p in PROMPTS if p["category"] == "compositional"][:2]
pmap = {p["id"]: p for p in PROMPTS}

def show_grid(ids, seed=0, title=""):
    fig, axes = plt.subplots(len(ids), 3, figsize=(10, 3.4 * len(ids)))
    for r, pid in enumerate(ids):
        for c, m in enumerate(["base", "run1", "run2"]):
            ax = axes[r, c]; ax.imshow(load_gen(m, pid, seed)); ax.set_xticks([]); ax.set_yticks([])
            if r == 0: ax.set_title({"base": "Base SD 1.5", "run1": "Run 1 (LoRA r4)", "run2": "Run 2 (LoRA r32)"}[m])
            if c == 0: ax.set_ylabel(f"{pid}\n{pmap[pid]['prompt'][:45]}", fontsize=8)
    plt.suptitle(title); plt.tight_layout(); plt.show()

show_grid(grid_ids, title="Base vs Run 1 vs Run 2 (seed 0)")

In [ ]:
# Upload all 120 images with metadata (prompt, seed, model) as a Hugging Face image dataset
if HF_TOKEN:
    gen_df.to_csv(GEN_DIR / "metadata.csv", index=False)     # imagefolder format: file_name + metadata columns
    create_repo(IMAGE_DATASET_REPO, repo_type="dataset", exist_ok=True)
    api.upload_folder(folder_path=str(GEN_DIR), repo_id=IMAGE_DATASET_REPO, repo_type="dataset",
                      commit_message="Part 2 generations: base, run1, run2 x 20 prompts x 2 seeds")
    print("pushed", f"https://huggingface.co/datasets/{IMAGE_DATASET_REPO}")

## 2.4 Evaluation
| Measurement | Definition |
|---|---|
| Primary | KID (in-domain generations vs test images; lower is better, noisy with 20 images) and CLIP score (prompt vs image) |
| Secondary | Diversity: mean LPIPS between the seed-0 and seed-1 image of each prompt |
| Human | Blinded final-run vs base comparison of 25 pairs, ≥3 raters, win rate |
| Efficiency | GPU-hours per training run (+ peak GPU memory) |

In [ ]:
del pipe; gc.collect(); torch.cuda.empty_cache()
from torchmetrics.image.kid import KernelInceptionDistance
from transformers import CLIPModel, CLIPProcessor
import lpips

def u8(paths, size=None):
    arr = []
    for p in paths:
        im = Image.open(p).convert("RGB")
        if size: im = im.resize((size, size), Image.BICUBIC)
        arr.append(torch.from_numpy(np.asarray(im)).permute(2, 0, 1))
    return torch.stack(arr)

real = u8(test_df.file, 299)
results = {}
id_ids = [p["id"] for p in PROMPTS if p["category"] == "in_domain"]
for m in ["base", "run1", "run2"]:
    fake_paths = [GEN_DIR / m / f"{pid}_s{s}.png" for pid in id_ids for s in GEN_SEEDS]
    fake = u8(fake_paths, 299)
    kid = KernelInceptionDistance(subset_size=min(len(fake), len(real)) // 2, subsets=100, normalize=False).to(device)
    for i in range(0, len(real), 32): kid.update(real[i:i + 32].to(device), real=True)
    kid.update(fake.to(device), real=False)
    torch.manual_seed(SEED)
    km, ks = kid.compute()
    results[m] = {"KID_mean": float(km), "KID_std": float(ks), "n_fake": len(fake), "n_real": len(real)}
    del kid; torch.cuda.empty_cache()
print(pd.DataFrame(results).T)

In [ ]:
# CLIP score = 100 * max(cos(image, prompt), 0) with openai/clip-vit-base-patch16, per image
clip_id = "openai/clip-vit-base-patch16"
clip_m = CLIPModel.from_pretrained(clip_id).to(device).eval(); clip_p = CLIPProcessor.from_pretrained(clip_id)

@torch.no_grad()
def clip_scores(paths, texts):
    out = []
    for i in range(0, len(paths), 16):
        ims = [Image.open(p).convert("RGB") for p in paths[i:i + 16]]
        inp = clip_p(text=texts[i:i + 16], images=ims, return_tensors="pt", padding=True, truncation=True).to(device)
        ie = torch.nn.functional.normalize(clip_m.get_image_features(pixel_values=inp.pixel_values), dim=-1)
        te = torch.nn.functional.normalize(clip_m.get_text_features(input_ids=inp.input_ids,
                                                                    attention_mask=inp.attention_mask), dim=-1)
        out += (100 * (ie * te).sum(-1).clamp(min=0)).tolist()
    return out

gen_df["clip_score"] = clip_scores([GEN_DIR / f for f in gen_df.file_name], list(gen_df.prompt))
cs = gen_df.pivot_table(index="model", columns="category", values="clip_score", aggfunc="mean")
cs["all"] = gen_df.groupby("model").clip_score.mean()
print(cs.round(2))

In [ ]:
# Diversity: LPIPS(AlexNet) between seed 0 and seed 1 for each prompt (higher = more diverse)
lp = lpips.LPIPS(net="alex", verbose=False).to(device)
def lp_t(p): return (torch.from_numpy(np.asarray(Image.open(p).convert("RGB"), dtype=np.float32) / 127.5 - 1)
                     .permute(2, 0, 1)[None].to(device))
div_rows = []
with torch.no_grad():
    for m in ["base", "run1", "run2"]:
        for p in PROMPTS:
            d = lp(lp_t(GEN_DIR / m / f"{p['id']}_s0.png"), lp_t(GEN_DIR / m / f"{p['id']}_s1.png")).item()
            div_rows.append({"model": m, "prompt_id": p["id"], "category": p["category"], "lpips": d})
div_df = pd.DataFrame(div_rows)
print(div_df.pivot_table(index="model", columns="category", values="lpips").round(3))
print("Least diverse prompts per model (possible repeated outputs):")
print(div_df.sort_values("lpips").groupby("model").head(3))

In [ ]:
# Results table (baseline = base SD 1.5, then both runs)
res = pd.DataFrame(results).T
res["CLIP_in_domain"] = cs["in_domain"]; res["CLIP_all"] = cs["all"]
res["LPIPS_diversity"] = div_df.groupby("model").lpips.mean()
res["gpu_hours_training"] = [0.0, train_metrics["run1"]["gpu_hours"], train_metrics["run2"]["gpu_hours"]]
res["peak_gpu_mem_GB_training"] = [np.nan, train_metrics["run1"]["peak_gpu_mem_GB"], train_metrics["run2"]["peak_gpu_mem_GB"]]
res["median_sec_per_image"] = gen_df.groupby("model").seconds.median()
res["nsfw_flagged"] = gen_df.groupby("model").nsfw_flagged.sum()
res = res.loc[["base", "run1", "run2"]]
res.round(4).to_csv(EVAL_DIR / "p2_results.csv")
gen_df.to_csv(EVAL_DIR / "p2_per_image_scores.csv", index=False)
display(res.round(4))

In [ ]:
# Sensitivity check: the safety checker replaces flagged images with black squares, which distorts KID/CLIP/LPIPS.
# Re-compute the metrics with flagged images excluded (main table above keeps them, as generated).
flagged = set(gen_df.loc[gen_df.nsfw_flagged, "file_name"])
print("Flagged (blacked-out) images:", len(flagged)); print(gen_df.groupby("model").nsfw_flagged.sum())
ok = gen_df[~gen_df.nsfw_flagged]
res_clean = {}
for m in ["base", "run1", "run2"]:
    fp = [GEN_DIR / f for f in ok[(ok.model == m) & (ok.category == "in_domain")].file_name]
    kid = KernelInceptionDistance(subset_size=max(2, len(fp) // 2), subsets=100, normalize=False).to(device)
    for i in range(0, len(real), 32): kid.update(real[i:i + 32].to(device), real=True)
    kid.update(u8(fp, 299).to(device), real=False)
    torch.manual_seed(SEED); km, ks = kid.compute(); del kid
    pairs_ok = div_df[(div_df.model == m) & ~div_df.prompt_id.apply(
        lambda pid: any(f"{m}/{pid}_s{s}.png" in flagged for s in GEN_SEEDS))]
    res_clean[m] = {"n_in_domain_used": len(fp), "KID_mean": float(km), "KID_std": float(ks),
                    "CLIP_in_domain": ok[(ok.model == m) & (ok.category == "in_domain")].clip_score.mean(),
                    "CLIP_all": ok[ok.model == m].clip_score.mean(), "LPIPS_diversity": pairs_ok.lpips.mean()}
res_clean = pd.DataFrame(res_clean).T
res_clean.round(4).to_csv(EVAL_DIR / "p2_results_excluding_nsfw_flagged.csv")
display(res_clean.round(4))

**Choose the final run** from the table and grids above (consider KID, in-domain CLIP score, diversity, and how
badly each run ignores out-of-domain prompts). Set `FINAL_RUN` below and explain the choice in the report.

In [ ]:
FINAL_RUN = "run1"   # <-- set to "run1" or "run2" after looking at the results
assert FINAL_RUN in RUNS

### Human evaluation — blinded comparison of 25 pairs (final run vs base)
The cell below makes 25 side-by-side images labelled **A** and **B** (left/right randomised with seed 266) and a
blank rating sheet. Each rater (both members + one person outside the team) copies the sheet to
`human_eval/p2_ratings_<name>.csv` and, for every pair, writes `A` or `B` in `better_style` (closer to the
Naruto domain style) and `better_prompt` (better match to the prompt). **Raters must not open
`p2_pairs_key.csv` until everyone has finished.** Then re-run the scoring cell.

In [ ]:
rng = random.Random(SEED)
cands = [(p["id"], s) for p in PROMPTS for s in GEN_SEEDS]
chosen = rng.sample(cands, 25)
pair_dir = HE_DIR / "p2_pairs"; pair_dir.mkdir(exist_ok=True)
key, sheet = [], []
for k, (pid, s) in enumerate(chosen, 1):
    final_left = rng.random() < 0.5
    left, right = (FINAL_RUN, "base") if final_left else ("base", FINAL_RUN)
    canvas = Image.new("RGB", (2 * RES + 30, RES + 70), "white")
    canvas.paste(load_gen(left, pid, s), (0, 70)); canvas.paste(load_gen(right, pid, s), (RES + 30, 70))
    dr = ImageDraw.Draw(canvas)
    dr.text((10, 8), f"Pair {k:02d}: {pmap[pid]['prompt'][:110]}", fill="black")
    dr.text((RES // 2, 40), "A", fill="black"); dr.text((RES + 30 + RES // 2, 40), "B", fill="black")
    canvas.save(pair_dir / f"pair_{k:02d}.png")
    key.append({"pair_id": k, "prompt_id": pid, "seed": s, "A": left, "B": right})
    sheet.append({"pair_id": k, "prompt": pmap[pid]["prompt"], "better_style": "", "better_prompt": ""})
pd.DataFrame(key).to_csv(HE_DIR / "p2_pairs_key.csv", index=False)
pd.DataFrame(sheet).to_csv(HE_DIR / "p2_rating_sheet_TEMPLATE.csv", index=False)
print(f"Wrote 25 pairs to {pair_dir} and the blank sheet to {HE_DIR/'p2_rating_sheet_TEMPLATE.csv'}")

In [ ]:
# Score the human evaluation (run after >= 3 rating files exist)
key_df = pd.read_csv(HE_DIR / "p2_pairs_key.csv")
rating_files = sorted(HE_DIR.glob("p2_ratings_*.csv"))
print("Rating files:", [f.name for f in rating_files])
if len(rating_files) >= 3:
    long = []
    for f in rating_files:
        r = pd.read_csv(f, dtype={"better_style": str, "better_prompt": str}).merge(key_df, on="pair_id")
        for crit in ["better_style", "better_prompt"]:
            ch = r[crit].str.strip().str.upper()
            assert ch.isin(["A", "B"]).all(), f"{f.name}: every {crit} must be A or B"
            winner = np.where(ch == "A", r["A"], r["B"])
            long += [{"rater": f.stem.replace("p2_ratings_", ""), "pair_id": pid, "criterion": crit,
                      "final_wins": w == FINAL_RUN} for pid, w in zip(r.pair_id, winner)]
    he = pd.DataFrame(long)
    per_rater = he.pivot_table(index="rater", columns="criterion", values="final_wins", aggfunc="mean")
    overall = he.groupby("criterion").final_wins.mean()
    majority = he.groupby(["criterion", "pair_id"]).final_wins.mean().gt(0.5).groupby("criterion").mean()
    print("Win rate of", FINAL_RUN, "vs base, per rater:\n", per_rater.round(3))
    print("Overall win rate (all votes):\n", overall.round(3))
    print("Majority-vote win rate (pairs):\n", majority.round(3))
    pd.concat({"overall": overall, "majority": majority}, axis=1).to_csv(EVAL_DIR / "p2_human_eval.csv")
else:
    print("Need at least 3 rating files (both members + 1 outside rater).")

### Failure modes
Out-of-domain and compositional prompts are where fine-tuned models usually fail (style leakage / prompt ignored,
attribute binding errors such as swapped colours). Pick **at least two** failure modes from these grids and from the
least-diverse list above and describe them with the example images in the report.

In [ ]:
show_grid([p["id"] for p in PROMPTS if p["category"] == "out_of_domain"], seed=0, title="Out-of-domain prompts")
show_grid([p["id"] for p in PROMPTS if p["category"] == "compositional"], seed=0, title="Compositional prompts")
for m in RUNS:
    worst = div_df[div_df.model == m].nsmallest(1, "lpips").iloc[0]
    fig, ax = plt.subplots(1, 2, figsize=(7, 3.6))
    for a, s in zip(ax, GEN_SEEDS):
        a.imshow(load_gen(m, worst.prompt_id, s)); a.set_title(f"{m} seed {s}"); a.axis("off")
    plt.suptitle(f"{m}: least diverse prompt {worst.prompt_id} (LPIPS={worst.lpips:.3f})"); plt.show()

## Safety (C4): license, NSFW filter, memorization check
* **License:** printed from the dataset card in 2.1. Naruto characters are copyrighted, so the adapters are released
  for non-commercial research only (stated in the model card). Base model license: CreativeML OpenRAIL-M.
* **NSFW filter:** the Stable Diffusion safety checker was enabled for all generations; flag counts are in the table.
* **Memorization:** for the 10 in-domain final-run images (seed 0), find the most similar training image by CLIP
  ViT-L/14 image-embedding cosine similarity and show the 3 closest pairs.

In [ ]:
mem_id = "openai/clip-vit-large-patch14"
mem_m = CLIPModel.from_pretrained(mem_id, torch_dtype=torch.float16).to(device).eval()
mem_p = CLIPProcessor.from_pretrained(mem_id)

@torch.no_grad()
def img_emb(paths):
    out = []
    for i in range(0, len(paths), 32):
        px = mem_p(images=[Image.open(p).convert("RGB") for p in paths[i:i + 32]], return_tensors="pt").pixel_values
        out.append(torch.nn.functional.normalize(mem_m.get_image_features(pixel_values=px.to(device, torch.float16)).float(), dim=-1))
    return torch.cat(out)

train_e = img_emb(list(train_df.file))
gen_paths = [GEN_DIR / FINAL_RUN / f"{pid}_s0.png" for pid in id_ids]
sim = img_emb(gen_paths) @ train_e.T
best_sim, best_idx = sim.max(dim=1)
mem_df = pd.DataFrame({"prompt_id": id_ids, "nearest_train_file": [train_df.file.iloc[i] for i in best_idx.tolist()],
                       "nearest_train_caption": [train_df.text.iloc[i] for i in best_idx.tolist()],
                       "clip_cosine": best_sim.cpu().numpy().round(4)}).sort_values("clip_cosine", ascending=False)
mem_df.to_csv(EVAL_DIR / "p2_memorization.csv", index=False)
display(mem_df)
fig, axes = plt.subplots(3, 2, figsize=(7, 10.5))
for r, (_, row) in enumerate(mem_df.head(3).iterrows()):
    axes[r, 0].imshow(load_gen(FINAL_RUN, row.prompt_id, 0)); axes[r, 0].set_title(f"generated {row.prompt_id}", fontsize=9)
    axes[r, 1].imshow(Image.open(row.nearest_train_file)); axes[r, 1].set_title(f"nearest train (cos={row.clip_cosine:.3f})", fontsize=9)
    for a in axes[r]: a.axis("off")
plt.suptitle("3 closest generated/training pairs"); plt.tight_layout(); plt.show()

## Model card for the final run (pushed to Hugging Face)
All numbers are filled in from the variables computed above. Edit the "Limitations" text in your own words.

In [ ]:
fm, fr = train_metrics[FINAL_RUN], res.loc[FINAL_RUN]
try:
    ds_license = DatasetCard.load(DATASET_ID).data.to_dict().get("license", "see dataset card")
except Exception:
    ds_license = "see dataset card"
card = f"""---
license: other
license_name: non-commercial-research-only
base_model: {BASE_MODEL}
datasets:
- {DATASET_ID}
tags: [stable-diffusion, lora, text-to-image, diffusers, data266]
---
# DATA 266 Lab 2 Team {TEAM} — {FINAL_RUN} ({fm['method']}) on Naruto BLIP captions

**Base model:** `{BASE_MODEL}` (512×512). **Adapter:** LoRA on UNet attention (`to_q,to_k,to_v,to_out.0`),
{fm['trainable_params']:,} trainable parameters ({fm['trainable_pct']}% of the UNet).

**Data:** `{DATASET_ID}` (dataset license: {ds_license}). After cleaning ({cleaning_report['corrupted_removed']} corrupted,
{cleaning_report['exact_duplicates_removed']} exact and {cleaning_report['near_duplicates_removed']} near duplicates removed),
{desc['n_train']} train / {desc['n_test']} test images (90/10 split, seed 266).

**Training:** {fm['steps']} steps, batch {fm['batch_size']}, AdamW lr {fm['learning_rate']}, cosine schedule with
{WARMUP_STEPS} warm-up steps, fp16 mixed precision, seed {SEED}; {fm['train_time_min']} min on {fm['gpu']}
({fm['gpu_hours']} GPU-h, peak {fm['peak_gpu_mem_GB']} GB).

**Results** (DPM-Solver++ 30 steps, CFG {GUIDANCE}; base SD 1.5 in brackets):
KID {fr.KID_mean:.4f} ± {fr.KID_std:.4f} [{res.loc['base'].KID_mean:.4f}],
CLIP score in-domain {fr.CLIP_in_domain:.2f} [{res.loc['base'].CLIP_in_domain:.2f}],
CLIP all prompts {fr.CLIP_all:.2f} [{res.loc['base'].CLIP_all:.2f}],
LPIPS diversity {fr.LPIPS_diversity:.3f} [{res.loc['base'].LPIPS_diversity:.3f}].
KID uses only 20 generated images and is noisy.

**Limitations:** TODO (your own words — e.g. style leaks into out-of-domain prompts, weak attribute binding,
BLIP captions do not name characters).

**License / intended use:** Naruto characters are copyrighted. This adapter is for **non-commercial research and
education only**. Generated images were filtered with the Stable Diffusion safety checker.

```python
from diffusers import StableDiffusionPipeline
import torch
pipe = StableDiffusionPipeline.from_pretrained("{BASE_MODEL}", torch_dtype=torch.float16).to("cuda")
pipe.load_lora_weights("{MODEL_REPOS[FINAL_RUN]}")
pipe("a man with orange hair and a headband").images[0]
```
"""
(OUT / FINAL_RUN / "final" / "README.md").write_text(card)
print(card)
if HF_TOKEN:
    api.upload_file(path_or_fileobj=str(OUT / FINAL_RUN / "final" / "README.md"), path_in_repo="README.md",
                    repo_id=MODEL_REPOS[FINAL_RUN], repo_type="model", commit_message="model card")
    print("model card pushed to", f"https://huggingface.co/{MODEL_REPOS[FINAL_RUN]}")

In [ ]:
# Links for links.txt
print("\n".join([f"Part 2 run1 model: https://huggingface.co/{MODEL_REPOS['run1']}",
                 f"Part 2 run2 model: https://huggingface.co/{MODEL_REPOS['run2']}",
                 f"Part 2 image dataset: https://huggingface.co/datasets/{IMAGE_DATASET_REPO}"]))